---
## 6. Enkodiranje kategoričkih varijabli

Za ML modele trebamo numeričke vrijednosti. Koristimo dvije strategije:
- **Label encoding** za ordinalne ili visoko-kardinalne varijable
- **One-hot encoding** za nisko-kardinalne (< 10 kategorija)

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

# --- LABEL ENCODING (visoka kardinalnost ili redoslijed nije bitan) ---
label_encode_cols = [
    'CustomerID',
    'Item Code',
    'Manufacturing Location Code',
    'Sales Channel - Internal',
    'Sales Channel - External',
    'Top Customer Group',    # OTHER=0, STAR=1
    'Intercompany',          # NO=0, YES=1
    'Make vs Buy',           # BUY=0, MANUFACTURED=1
]

le_dict = {}  # spremamo enkodere za inverse_transform kasnije

for col in label_encode_cols:
    if col in df_encoded.columns:
        le = LabelEncoder()
        df_encoded[col + '_enc'] = le.fit_transform(df_encoded[col].astype(str))
        le_dict[col] = le
        print(f'{col}: {df_encoded[col].nunique()} kategorija → label encoded')

# --- ONE-HOT ENCODING (niska kardinalnost) ---
ohe_cols = [
    'Manufacturing Region',   # North America, Asia, Europe
    'Customer Region',        # North America, Asia, Europe  
    'Product family',         # PF000, PF001, PF002
]

for col in ohe_cols:
    if col in df_encoded.columns:
        dummies = pd.get_dummies(df_encoded[col], prefix=col, drop_first=False)
        df_encoded = pd.concat([df_encoded, dummies], axis=1)
        print(f'{col}: one-hot → {dummies.shape[1]} novih stupaca')

print(f'\nShape nakon enkodiranja: {df_encoded.shape}')

In [ ]:
# Customer industry enkodiranje (IC000, IC001... → numerički)
# Izvuci broj iz stringa
if 'Customer industry' in df_encoded.columns:
    df_encoded['customer_industry_num'] = (
        df_encoded['Customer industry']
        .str.extract(r'(\d+)')
        .astype(float)
    )
    print('Customer industry → customer_industry_num ✓')
    print(df_encoded['customer_industry_num'].value_counts())

---
## 7. Feature Engineering — izvedene značajke

Ovo je ključni korak — kreiramo varijable koje modelu daju kontekst o kupcu i ponašanju.

In [ ]:
# 7a. Vremenski featuri iz Invoice Date
df_encoded['invoice_year'] = df_encoded['Invoice Date'].dt.year
df_encoded['invoice_month'] = df_encoded['Invoice Date'].dt.month
df_encoded['invoice_quarter'] = df_encoded['Invoice Date'].dt.quarter
df_encoded['invoice_dayofweek'] = df_encoded['Invoice Date'].dt.dayofweek

print('Vremenski featuri dodani ✓')

In [ ]:
# 7b. Customer tenure — koliko dugo je kupac aktivan (u danima)
reference_date = df_encoded['Invoice Date'].max()
df_encoded['customer_tenure_days'] = (
    reference_date - df_encoded['Customer First Invoice Date']
).dt.days

print('Customer tenure izračunat ✓')
print(df_encoded['customer_tenure_days'].describe())

In [ ]:
# 7c. Revenue po liniji (qty * price)
df_encoded['line_revenue'] = (
    df_encoded['Invoiced qty (shipped)'] * df_encoded['Invoiced price']
)

# 7d. Gross profit po liniji
df_encoded['line_gross_profit'] = df_encoded['line_revenue'] * df_encoded['GM%']

print('Revenue i gross profit featuri dodani ✓')

In [ ]:
# 7e. Customer-level aggregate featuri
# Ovo je srž za customer segmentaciju!
customer_agg = df_encoded.groupby('CustomerID').agg(
    total_revenue=('line_revenue', 'sum'),
    total_orders=('Invoice #', 'nunique'),
    total_items=('Item Code', 'nunique'),
    avg_order_value=('line_revenue', 'mean'),
    avg_gm=('GM%', 'mean'),
    n_product_families=('Product family', 'nunique'),
    n_product_groups=('Product group', 'nunique'),
    customer_tenure_days=('customer_tenure_days', 'first'),
    customer_region=('Customer Region', 'first'),
    customer_industry=('Customer industry', 'first'),
    top_customer_group=('Top Customer Group', 'first'),
).reset_index()

print(f'Customer aggregate tablica: {customer_agg.shape}')
display(customer_agg.head())

In [ ]:
# 7f. Vizualizacija distribucija ključnih featura
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

customer_agg['total_revenue'].apply(lambda x: np.log1p(x)).hist(
    bins=40, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Log(Total Revenue) po kupcu')

customer_agg['total_orders'].hist(bins=40, ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Broj narudžbi po kupcu')

customer_agg['total_items'].hist(bins=40, ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Broj unikatnih proizvoda po kupcu')

customer_agg['avg_gm'].hist(bins=40, ax=axes[1,0], color='salmon')
axes[1,0].set_title('Prosječni GM% po kupcu')

customer_agg['customer_tenure_days'].hist(bins=40, ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Customer Tenure (dani)')

customer_agg['n_product_groups'].hist(bins=30, ax=axes[1,2], color='green', alpha=0.7)
axes[1,2].set_title('Broj product grupe po kupcu')

plt.suptitle('Distribucije ključnih customer featura', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 7g. Product co-occurrence matrica (osnova za cross-sell)
# Grupiraj po Invoice # — koji proizvodi se pojavljuju na istoj fakturi?
invoice_items = df_encoded.groupby('Invoice #')['Product group'].apply(
    lambda x: list(x.dropna().unique())
).reset_index()
invoice_items.columns = ['invoice_id', 'product_groups']

# Filtriraj fakture s više od jednog produkta (cross-sell prilike)
multi_product = invoice_items[invoice_items['product_groups'].apply(len) > 1]
print(f'Fakture s 1 produktom: {(invoice_items["product_groups"].apply(len) == 1).sum():,}')
print(f'Fakture s 2+ produkata: {len(multi_product):,}')
print(f'% multi-product faktura: {len(multi_product)/len(invoice_items)*100:.1f}%')
display(multi_product.head(5))

---
## 8. Korelacijska analiza numeričkih featura

In [ ]:
num_features = [
    'Invoiced qty (shipped)', 'Invoiced price', 'Cost of part',
    'GM%', 'line_revenue', 'customer_tenure_days',
    'invoice_month', 'invoice_quarter'
]

existing_num = [c for c in num_features if c in df_encoded.columns]
corr = df_encoded[existing_num].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Korelacijska matrica numeričkih featura')
plt.tight_layout()
plt.show()

---
## 9. Finalni dataset — spremi za modeliranje

In [ ]:
# Finalni pregled
print('=== FINALNI DATASET ===')
print(f'Shape: {df_encoded.shape}')
print(f'Nullovi: {df_encoded.isnull().sum().sum()}')
print(f'\nStupci:')
for col in df_encoded.columns:
    print(f'  {col}: {df_encoded[col].dtype}')

In [ ]:
# Spremi očišćeni dataset
df_encoded.to_csv('../data/processed/data_clean.csv', index=False)
customer_agg.to_csv('../data/processed/customer_features.csv', index=False)
invoice_items.to_csv('../data/processed/invoice_items.csv', index=False)

print('Spremljeno ✓')
print('  → data/processed/data_clean.csv')
print('  → data/processed/customer_features.csv')
print('  → data/processed/invoice_items.csv')

---
## Sažetak — što smo napravili

| Korak | Akcija |
|-------|--------|
| Enkodiranje | Label encoding za visoko-kardinalne, OHE za nisko-kardinalne |
| Novi featuri | Tenure, revenue, gross profit, vremenski featuri |
| Agregati | Customer-level tablica za segmentaciju |
| Co-occurrence | Invoice-items tablica za market basket analysis |